# z2 — 30-minute Colab run (NaNROUTER diverse + Gutenberg)

**Runtime → Change runtime type → T4 GPU** before running.

This trains the 84.86M v2 hybrid for 30 minutes on a mixed corpus:

- `corpora-paragraphs-0010`
- `corpora-short-stories-0004`
- `corpora-sentences-0010`
- `hw5-gutenberg-c256-g1` (21-book Gutenberg reference)

Artifacts are saved to Google Drive, not ephemeral `/content`.


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q tokenizers zstandard

In [ ]:
!git clone --depth 1 https://github.com/shemshallah/NaNROUTER /content/NaNROUTER
!git clone --depth 1 https://github.com/shemshallah/nanlm_experimental /content/nanlm_experimental

## Persist artifacts

Mount Drive. If you skip this, everything is lost when the Colab runtime ends.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nanlmv3_z2'
import os
os.makedirs(OUT, exist_ok=True)
print(OUT)

## Run the 30-minute job

The script downloads the corpus releases, builds a mixed 200MB corpus, reuses
the z1 BPE tokenizer, and trains for 1,800 seconds with time-cosine LR to zero
and EMA weights.


In [ ]:
import os
CKPT = f"{OUT}/best.pt"
print('checkpoint:', CKPT, 'exists:', os.path.exists(CKPT))


In [ ]:
!python /content/nanlm_experimental/z2/train_colab.py --minutes 30 --out "$OUT" --resume-best


## Show result


In [ ]:
import json
print(json.dumps(json.load(open(f"{OUT}/result.json")), indent=2, sort_keys=True))

## Optional continuation

To start from a previous `.pt`, pass a direct URL:

```python
CHECKPOINT_URL = "https://github.com/.../best.pt"
!python /content/nanlm_experimental/z2/train_colab.py \
  --minutes 30 --out "$OUT" --checkpoint-url "$CHECKPOINT_URL"
```
